# Bronze -> Silver: visão consolidada

ADR [0022](../../docs/adr/0022-notebooks-de-diagnostico-medallion-separados-da-narrativa-do-tcc.md)
(issue [#128](https://github.com/Vini0606/Tecnicas-de-Ciencia-de-Dados-em-dados-do-Instagram/issues/128)).
Volumetria, completude e o que a limpeza Silver muda em relação à Bronze, para as 4 tabelas Bronze +
6 Silver de uma vez -- estes dois estágios são ingestão/limpeza, não modelagem, então não têm
profundidade individual suficiente para justificar um notebook por tabela (ver ADR 0022, opções
consideradas).

`ugc_mentions` (Bronze e Silver) é coletada em toda Coleta, como última etapa tolerante a falha da
extração (ADR 0034; ADR 0020 Ficha 8 / issue #93). Não tem par direto de `profiles`/`posts`/`reels`
(o autor é um terceiro, não o governador), mas cabe no mesmo notebook consolidado pelo mesmo
raciocínio: pouca profundidade analítica própria de ingestão/limpeza (o domínio de UGC que merece
profundidade fica em `gold_ugc.ipynb`). As regras de cada cleaner estão no README, seção 2.3.

In [1]:
import sys
import os

# Adiciona o diretório raiz do projeto ao sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
from dotenv import load_dotenv

from config import settings
from src.repositories.delta_repository import DeltaRepository
from src.analysis.medallion_diagnostics import completeness_summary, count_duplicate_rows

load_dotenv()
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)

## Carga

Tudo pelo `DeltaRepository`, a única porta de leitura das três camadas (issue #313): `bronze_dir`
dá acesso às quatro tabelas Bronze (`load_bronze_*`) e os acessores `load_silver_*` cobrem as Silver
que não tinham acessor próprio. Atenção aos nomes legados sem prefixo: `load_profiles()` lê a Gold
`governor_engagement` e `load_comments()` prefere a Gold `governor_sentiment`; aqui usamos os
acessores explícitos de Silver.

In [2]:
repo = DeltaRepository(
    gold_dir=settings.GOLD_DIR,
    silver_dir=settings.SILVER_DIR,
    bronze_dir=settings.BRONZE_DIR,
)

tabelas = {
    'bronze.instagram_profiles': repo.load_bronze_profiles(),
    'bronze.instagram_posts': repo.load_bronze_posts(),
    'bronze.instagram_reels': repo.load_bronze_reels(),
    'silver.profiles_clean': repo.load_silver_profiles(),
    'silver.posts_clean': repo.load_posts(),
    'silver.reels_clean': repo.load_reels(),
    'silver.comments_clean': repo.load_silver_comments(),
    'silver.governors_metadata': repo.load_governors_metadata(),
    'bronze.ugc_mentions': repo.load_bronze_ugc_mentions(),
    'silver.ugc_mentions': repo.load_silver_ugc_mentions(),
}

## Volumetria e duplicatas

In [3]:
resumo = pd.DataFrame({
    'tabela': list(tabelas.keys()),
    'n_linhas': [len(df) for df in tabelas.values()],
    'n_colunas': [df.shape[1] for df in tabelas.values()],
    'n_linhas_duplicadas': [count_duplicate_rows(df) for df in tabelas.values()],
})
resumo

,tabela,n_linhas,n_colunas,n_linhas_duplicadas
0,bronze.instagram_profiles,26,18,0
1,bronze.instagram_posts,9637,20,0
2,bronze.instagram_reels,7202,21,0
3,silver.profiles_clean,26,14,0
4,silver.posts_clean,8437,18,0
5,silver.reels_clean,5904,20,0
6,silver.comments_clean,52464,12,0
7,silver.governors_metadata,26,7,0
8,bronze.ugc_mentions,16849,18,0
9,silver.ugc_mentions,16824,14,0


## Completude por tabela

Só colunas com pelo menos um nulo aparecem -- uma tabela sem saída abaixo do seu nome está
100% completa nas colunas que tem.

In [4]:
for nome, df in tabelas.items():
    resumo_completude = completeness_summary(df)
    colunas_com_nulo = resumo_completude[resumo_completude['n_nulos'] > 0]
    if not colunas_com_nulo.empty:
        print(f'--- {nome} ---')
        display(colunas_com_nulo)

--- bronze.instagram_profiles ---


,coluna,dtype,n_nulos,pct_nulos,n_unicos
8,igtvVideoCount,float64,26,100.00,0
12,hasChannel,object,26,100.00,0
13,joinedRecently,object,26,100.00,0


--- bronze.instagram_posts ---


,coluna,dtype,n_nulos,pct_nulos,n_unicos
3,inputUrl,object,2,0.02,26
10,videoViewCount,float64,9637,100.00,0
11,videoPlayCount,float64,2962,30.74,6581
12,videoDuration,float64,2967,30.79,4978
13,locationName,object,6513,67.58,776


--- bronze.instagram_reels ---


,coluna,dtype,n_nulos,pct_nulos,n_unicos
3,inputUrl,object,2,0.03,26
6,videoViewCount,float64,7202,100.00,0
12,isSponsored,object,7202,100.00,0
13,isCommentsDisabled,object,2,0.03,2
15,latestComments,object,2,0.03,7178
16,transcript,object,7202,100.00,0


... (6 outputs left out)

--- bronze.ugc_mentions ---


,coluna,dtype,n_nulos,pct_nulos,n_unicos
0,id,object,12,0.07,16824
1,shortCode,object,12,0.07,16824
2,type,object,12,0.07,3
3,caption,object,12,0.07,15759
4,mentions,object,12,0.07,2471
5,taggedUsers,object,19,0.11,15032
6,likesCount,float64,12,0.07,2253
7,commentsCount,float64,12,0.07,746
8,videoPlayCount,float64,6130,36.38,7831
9,timestamp,object,12,0.07,16779


--- silver.ugc_mentions ---


,coluna,dtype,n_nulos,pct_nulos,n_unicos
4,governor_username,object,11,0.07,17
9,videoPlayCount,float64,6114,36.34,7831


## O que a limpeza Silver muda em relação à Bronze

Para os três pares com correspondência direta (`profiles`, `posts`, `reels`): diferença de colunas
(o que a limpeza remove/deriva) e diferença de contagem de linhas (o que é descartado, ex.: perfis
sem dado suficiente para conformar o schema Silver).

In [5]:
pares_bronze_silver = {
    'profiles': ('bronze.instagram_profiles', 'silver.profiles_clean'),
    'posts': ('bronze.instagram_posts', 'silver.posts_clean'),
    'reels': ('bronze.instagram_reels', 'silver.reels_clean'),
    'ugc_mentions': ('bronze.ugc_mentions', 'silver.ugc_mentions'),
}

linhas_comparacao = []
for entidade, (nome_bronze, nome_silver) in pares_bronze_silver.items():
    df_bronze = tabelas[nome_bronze]
    df_silver = tabelas[nome_silver]
    linhas_comparacao.append({
        'entidade': entidade,
        'linhas_bronze': len(df_bronze),
        'linhas_silver': len(df_silver),
        'linhas_descartadas': len(df_bronze) - len(df_silver),
        'colunas_removidas_pela_limpeza': sorted(set(df_bronze.columns) - set(df_silver.columns)),
        'colunas_derivadas_pela_limpeza': sorted(set(df_silver.columns) - set(df_bronze.columns)),
    })

pd.DataFrame(linhas_comparacao)

,entidade,linhas_bronze,linhas_silver,linhas_descartadas,colunas_removidas_pela_limpeza,colunas_derivadas_pela_limpeza
0,profiles,26,26,0,"[_raw, _source, hasChannel, igtvVideoCount, jo...",[_source_layer]
1,posts,9637,8437,1200,"[_raw, _source, latestComments, locationName, ...","[Tipo, _source_layer, data_hora, hashtags, typ..."
2,reels,7202,5904,1298,"[_raw, _source, latestComments, timestamp, typ...","[Tipo, Total de Engajamento, _source_layer, da..."
3,ugc_mentions,16849,16824,25,"[_raw, _source, mentions, ownerFullName, owner...","[_source_layer, authorUsername, data_hora, gov..."


## Nota de interpretação

Comparar `linhas_descartadas` acima com os motivos de descarte que os cleaners aplicam (README,
seção 2.3): linha sem `id`, governador fora de `governadores.xlsx`, `id` repetido dentro da Coleta
e `timestamp` ilegível. Cada descarte gera um warning com contagem no log da Coleta -- é lá que se
confere o motivo antes de assumir que uma queda grande é bug. Em posts e reels, o `id` repetido é
o motivo mais comum: um Reel também aparece no grid do perfil, então o scraper de posts e o de
reels capturam a mesma publicação. Se `linhas_descartadas` crescer muito além disso numa Coleta
futura, vale investigar a extração (Bronze) antes de suspeitar da limpeza (Silver).

`ugc_mentions` é o único par onde `linhas_descartadas` não se compara ao universo de ~27
governadores -- o volume bruto depende de quantos terceiros marcaram os perfis no Recorte (actor
`apify/instagram-tagged-scraper`), não do número de perfis monitorados. Além disso,
`governor_username` nulo na Silver não é um descarte (a linha permanece, só sem correlação
resolvida) -- ver completude e análise de covariáveis específica em `gold_ugc.ipynb`.